# CAF Remote Computing tutorial using Globus

This tutorial notebook shows how to run computations and agents on remote HPC systems using Globus Compute and Academy. 

Globus Compute is a managed computing service. In addition to providing a simple Python/REST API for sending tasks to remote computers, it also provides a ``fire-and-forget'' computing model in which a managed cloud service takes responsibility for orchestrating execution of tasks on a remote endpoint. Users can retrieve results after completiion via the cloud service. 

This tutorial is configured to use the Globus Compute endpoint hosted by ALCF on the [*Polaris*](https://www.alcf.anl.gov/polaris) computer.  You can also set up your own endpoint on resources to which you have access by following the [Globus Compute documentation](https://globus-compute.readthedocs.io/en/latest/endpoints/endpoints.html).  Globus Compute endpoints can be deployed on many cloud platforms, clusters with batch schedulers (e.g., Slurm, PBS), Kubernetes, or on a local PC.  After configuring an endpoint you can use it in this tutorial by simply setting the `endpoint_id` below.



### Configuration

We first create a Globus Compute executor that we will use to submit and manage tasks.  The executor provides a simple asynchronous Python interface.  Note: the first time you create the executor you will need to authenticate with Globus Auth. To use the Polaris endpoint you must authenticate using an ALCF identity.  

Globus Compute Multi User Endpoints spawn a unique endpoint process for each user and subsequently
allow task exection. The user endpoint is configured according to the ``user_endpoint_config`` argument.  To use Polaris you must specify: 1) our account and 2) the queue to use. You can optionally specify a ``config_key`` to configure the execution environment. In this case, we will load a shared UV environment with various packages preinstalled. If you want to install additional tools, you will need to create a custom environment.

In [1]:
from globus_compute_sdk import Executor
from globus_compute_sdk.serialize import ComputeSerializer, CombinedCode

compute_endpoint = '9a947ba5-f537-4681-acf3-cc66485aadec' # Polaris Globus Compute endpoint

POLARIS_CONFIG = {
    'account': 'AuroraGPT',
    'queue': 'debug',
    'config_key': 'source /home/yadunand/setup_openmm.sh'
}

serializer = ComputeSerializer(strategy_code=CombinedCode())

gce = Executor(endpoint_id=compute_endpoint, 
               user_endpoint_config=POLARIS_CONFIG, 
               serializer=serializer)

### 1. Running a function

Globus Compute executes Python "functions". These functions may be pure Python code or may wrap binaries or command line invocations. 

In [2]:
def hello_world(name: str):
    import platform
    return f"Hello {name} from Polaris\n{list(platform.uname())}"

future = gce.submit(hello_world, "ModCon")

# Globus compute implements an asynchronous model. 
# Block and wait for the result
print(future.result())  

Hello ModCon from Polaris
['Linux', 'x3001c0s19b0n0', '6.4.0-150600.23.73-default', '#1 SMP PREEMPT_DYNAMIC Tue Oct  7 08:43:02 UTC 2025 (46f6a23)', 'x86_64', 'x86_64']


/Users/yadu/src/caf_hackathon_2026/test_py3.13/lib/python3.13/site-packages/globus_compute_sdk/sdk/client.py:315: UserWarning: 
Environment differences detected between local SDK and endpoint 37ee24f6-62b7-eb53-eaa2-dee4b0e134a3 workers:
	    SDK: Python 3.13.5/Dill 0.3.9
	Workers: Python 3.13.7/Dill 0.3.9
This may cause serialization issues.  See https://globus-compute.readthedocs.io/en/latest/sdk/executor_user_guide.html#avoiding-serialization-errors for more information.
  warnings.warn(check_result, UserWarning)


You can also run shell functions by passing arguments to the Globus Compute ``ShellFunction``.

In [3]:
from globus_compute_sdk import ShellFunction

bf = ShellFunction("echo '{message}'")
future = gce.submit(bf, message='Hello World')
shell_result = future.result()  # ShellFunctions return ShellResults
print(shell_result.stdout)

Hello World



### 2. Policy-based function routing

A common pattern we see across model teams is the need to dynamically choose which remote computations (e.g., simulations) should be run based on a decision policy (e.g., a LLM).

In this example, we have three functions to compute: prime numbers, molecular energy, and molecular dynamics trajectories. We use an LLM to decide which function to run. 

To remove dependencies and expedite the demo we use mock implementations of the energy and trajectory functions.

In [4]:
from functools import wraps
def gce_wrapper(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        future = gce.submit(func, *args, **kwargs)
        return future.result()
    return wrapper

@gce_wrapper
def compute_primes(n: int = 1000) -> dict:
    """Count prime numbers up to n."""
    def is_prime(x):
        if x < 2:
            return False
        for i in range(2, int(x**0.5) + 1):
            if x % i == 0:
                return False
        return True

    primes = [x for x in range(2, n + 1) if is_prime(x)]
    return {
        "task": "prime_count",
        "n": n,
        "count": len(primes),
        "largest": primes[-1] if primes else None,
        "first_10": primes[:10]
    }

@gce_wrapper
def compute_energy(molecule: str = "H2O") -> dict:
    """Simulate molecular energy calculation."""
    import random
    # Mock responses - real version would use PySCF or similar
    energies = {"H2O": -76.4, "CO2": -188.5, "CH4": -40.5,"N2": -109.5}
    base_energy = energies.get(molecule, -50.0) + random.uniform(-0.1, 0.1)
    return {
        "task": "energy_calculation",
        "molecule": molecule,
        "energy_hartree": base_energy,
        "method": "surrogate_dft",
        "basis": "cc-pVDZ"
    }

@gce_wrapper
def compute_trajectory(n_atoms: int = 27, n_steps: int = 100) -> dict:
    """Simulate MD trajectory."""
    import random
    temperatures = [300 + random.uniform(-20, 20) for _ in range(n_steps)]
    return {
        "task": "md_simulation",
        "n_atoms": n_atoms,
        "n_steps": n_steps,
        "avg_temperature": sum(temperatures) / len(temperatures),
        "final_temperature": temperatures[-1],
    }


We now implement the decision policy using an OpenAI compatible LLM API. 

Follow these instructions to obtain an API key for the ALCF inference service. Or insert a key, model name, and URL for another OpenAI compatible service. 

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
import json
import random

requests=["How many prime numbers are there below 500?",
        "What is the potential energy of a water molecule",
        "What is the MD trajectory of water molecules"]

api_key= ''
model_name = 'openai/gpt-oss-120b' 
base_url = 'https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1'
        
llm = ChatOpenAI(
    model=model_name,
    api_key=api_key,
    base_url=base_url,
)

CHOOSER_PROMPT = """You are a task router. Given a user request, choose which computation to run.

Available computations:
1. "primes" - Count prime numbers (for math/number questions)
2. "energy" - Calculate molecular energy (for chemistry/molecule questions)
3. "trajectory" - Run MD simulation (for physics/dynamics/trajectory questions)
"""

tools = [compute_primes, compute_energy, compute_trajectory]
agent = create_agent(
    llm, 
    tools=tools,
    system_prompt=CHOOSER_PROMPT,
)

# Run the agent
request=random.choice(requests)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": request}]},
    stream_mode='updates',
    version="v2"
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

# LLM-Guided Drug Discovery Demo

This demo showcases a full computational drug discovery pipeline that combines LLM reasoning with cheminformatics and molecular simulation. The core computational components are launched as Academy Agents on Polaris@ALCF.

The pipeline has five stages:

**1. Candidate Generation** — An LLM is prompted as a medicinal chemist to generate drug-like SMILES strings targeting a specified protein (e.g. COX-2, EGFR), along with a chemical rationale for each scaffold.

**2. Validation & Featurization** — RDKit validates each SMILES, generates a 3D conformer, and computes drug-likeness descriptors: molecular weight, logP, TPSA, H-bond donors/acceptors, and Lipinski compliance.

**3. Binding Energy Estimation** — MMFF94 vacuum minimization via RDKit computes a strain energy for each ligand (kcal/mol). Lower strain energy indicates a geometrically relaxed, binding-ready molecule. GPU acceleration via OpenMM can be swapped in for larger systems.

**4. Lead Selection** — An LLM reasons over the full descriptor table — balancing strain energy, drug-likeness, and chemical novelty — and selects a lead candidate with a written justification.

**5. Visualization** — A matplotlib dashboard displays 2D structures of all candidates with their key properties, along side a strain energy bar chart highlighting the lead.


In [5]:
import asyncio
import logging

# This demo uses Academy to launch agents on Polaris
import academy
from academy.manager import Manager
from academy.agent import Agent, action
from academy.handle import Handle
from academy.exchange.cloud import HttpExchangeFactory
from academy.logging import init_logging

# We use Globus Compute to launch Academy Agents on Polaris
from globus_compute_sdk import Executor as GlobusComputeExecutor

print(academy.__version__)

0.4.0


## Get tokens for the Argonne Inference Endpoint. 

In this step we will use the `inference_auth_token` module to provision an access token
from Argonne's Inference Endpoint.
This step will emit a Globus Auth URL that will take you to an authentication page. 

**NOTE**: You need to auth an ALCF/UChicago account to use the Argonne Inference Endpoint

In [6]:
from inference_auth_token import get_access_token
api_key = get_access_token()

## DrugReasoner

The DrugReasoner agent loads an LLM at Agent startup and uses this LLM for reasoning about Drug candidates.
Here we implement `generate_candidates` and `select_lead`.

In [7]:
class DrugReasoner(Agent):
    """DrugReasoner agent uses an LLM to generate and compare Drug candidates"""
    
    def __init__(self, api_key: str, model_name: str='openai/gpt-oss-120b', 
                 base_url='https://inference-api.alcf.anl.gov/resource_server/sophia/vllm/v1'):
        super().__init__()
        self.model_name = model_name
        self.base_url = base_url
        self.api_key = api_key
        self.llm = None
        
        
    async def agent_on_startup(self):
        """Load LLM object on agent startup """
        from langchain_openai import ChatOpenAI
        
        self.llm = ChatOpenAI(
            model=self.model_name,
            api_key=self.api_key,
            base_url=self.base_url,
        )


    def _parse_json_response(self, text: str):
        """Strip markdown fences (if any) and parse JSON."""
        import json
        
        cleaned = text.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("\n", 1)[-1]
        if cleaned.endswith("```"):
            cleaned = cleaned.rsplit("```", 1)[0]
        return json.loads(cleaned.strip())
        
    @action
    async def generate_candidates(self, target_description: str, n: int = 5) -> list[dict]:
        """Ask the LLM to generate candidate drug-like SMILES for a target."""
        
        prompt = f"""You are a medicinal chemist AI.
Generate {n} small-molecule drug candidates targeting: {target_description}

Rules:
- Each molecule must be drug-like (Lipinski's Rule of Five)
- Prefer known pharmacophores for this target class
- Return ONLY valid JSON, no preamble, no markdown fences

Return a JSON array of objects with these fields:
  "smiles"      : valid SMILES string
  "name"        : short descriptive name
  "rationale"   : one sentence explaining why this scaffold is promising

Example format:
[
  {{"smiles": "CCO", "name": "ethanol", "rationale": "Simple example."}}
]
"""
        try:
            response = self.llm.invoke(prompt)
            candidates = self._parse_json_response(response.content)
        except Exception as e:
            return f"Failed to connect to inference endpoint with {self.api_key}: error:{e}"
        
        return candidates
    
    @action
    async def select_lead(self, candidates: list[dict]) -> dict:
        """Pick the best candidate"""
        import json
        summaries = [
            {
                "name":                      c["name"],
                "smiles":                    c["smiles"],
                "rationale":                 c["rationale"],
                "mw":                        c.get("mw"),
                "logp":                      c.get("logp"),
                "tpsa":                      c.get("tpsa"),
                "lipinski_ok":               c.get("lipinski_ok"),
                "strain_energy_kcal_mol":    c.get("strain_energy"),   # kcal/mol not kJ/mol
            }
            for c in candidates if "strain_energy" in c
        ]
        
        prompt = f"""You are an expert computational medicinal chemist.
Below are drug candidates with computed properties and strain energies.
Strain energies are in kcal/mol computed via MMFF94 vacuum minimization.
Lower strain energy indicates a more geometrically relaxed, binding-ready ligand.

Pick the single best lead candidate, balancing:
- Low strain energy (flexible, ready to bind)
- Good drug-likeness (Lipinski compliant, low TPSA)
- Chemical novelty and rationale quality

Return ONLY valid JSON with fields:
  "lead_name"  : name of chosen candidate
  "reasoning"  : 2-3 sentence explanation of the choice

Candidates:
{json.dumps(summaries, indent=2)}
"""
        response = self.llm.invoke(prompt)
        lead = self._parse_json_response(response.content)
        return lead

## DrugValidator

We implement `DrugValidator` an academy agent that uses RDKit to identify chemical features and run a basic simulation to calculate Binding Energy which we will use as a proxy for the Drug's viability.

This agent implements two actions: `validate_and_featurize` and `estimate_binding_energy`

In [8]:
class DrugValidator(Agent):
    
    def __init__(self, ):
        super().__init__()
        
    @action
    async def validate_and_featurize(self, candidates: list[dict]) -> list[dict]:
        """Validate SMILES and compute drug-likeness descriptors."""
        from rdkit import Chem
        from rdkit.Chem import AllChem, Descriptors
        from rdkit.Chem import Draw
        valid = []
        for c in candidates:
            mol = Chem.MolFromSmiles(c["smiles"])
            if mol is None:
                continue

            mol = Chem.AddHs(mol)
            result = AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
            if result != 0:
                continue
            AllChem.MMFFOptimizeMolecule(mol)

            c["mol"]        = mol
            c["mw"]         = Descriptors.MolWt(mol)
            c["logp"]       = Descriptors.MolLogP(mol)
            c["hbd"]        = Descriptors.NumHDonors(mol)
            c["hba"]        = Descriptors.NumHAcceptors(mol)
            c["tpsa"]       = Descriptors.TPSA(mol)
            c["n_atoms"]    = mol.GetNumAtoms()
            c["lipinski_ok"] = (
                c["mw"] <= 500 and c["logp"] <= 5 and
                c["hbd"] <= 5  and c["hba"] <= 10
            )
            valid.append(c)

        return valid


    # ── Step 3: Binding energy proxy ───────────────────────
    @action
    async def estimate_binding_energy(self, candidates: list[dict]) -> list[dict]:
        """
        Estimates binding energy proxy via MMFF94 vacuum minimization using
        RDKit. Computes strain energy (energy before minimization - energy after)
        as a proxy for ligand flexibility / binding readiness.
        """
        from rdkit import Chem
        from rdkit.Chem import AllChem

        def mmff_strain_energy(mol_start):
            props = AllChem.MMFFGetMoleculeProperties(mol_start)
            if props is None:
                raise ValueError("MMFF94 parameterization failed")

            ff_before = AllChem.MMFFGetMoleculeForceField(mol_start, props)
            if ff_before is None:
                raise ValueError("MMFF94 force field setup failed")
            e_before = ff_before.CalcEnergy()

            AllChem.MMFFOptimizeMolecule(mol_start)

            props_after = AllChem.MMFFGetMoleculeProperties(mol_start)
            ff_after    = AllChem.MMFFGetMoleculeForceField(mol_start, props_after)
            if ff_after is None:
                raise ValueError("MMFF94 force field setup failed after minimization")
            e_after = ff_after.CalcEnergy()

            return round(e_before, 2), round(e_after, 2), round(e_before - e_after, 2)

        for c in candidates:
            try:
                mol = c["mol"]

                # Re-embed to get a fresh starting conformer before minimization
                mol_start = Chem.AddHs(Chem.RemoveHs(mol))
                result    = AllChem.EmbedMolecule(mol_start, AllChem.ETKDGv3())
                if result != 0:
                    raise ValueError("3D embedding failed")

                e_before, e_after, strain = mmff_strain_energy(mol_start)

                c["e_before"]      = e_before
                c["e_after"]       = e_after
                c["strain_energy"] = strain

            except Exception as ex:
                import traceback
                print(f"  ✗ Failed for {c['name']}: {ex}\n{traceback.format_exc()}")
                c["e_before"]      = None
                c["e_after"]       = None
                c["strain_energy"] = float("inf")
                c["exception"]     = str(ex)

        return candidates

# Launch Agents

Now that we have implemented the agents, we will use the Globus Compute Endpoint on Polaris to go through the 
following steps:
1. Generate Candidates based on a user prompt
2. Featurize the candidate molecules
3. Calculate Binding Energy as a proxy for drug viability/suitability
4. Select lead candidate

In [11]:
async def drug_finder(prompt: str) -> tuple[list[dict], dict]:
    
    # We will use the Polaris Globus Compute Endpoint for launching the two agents    
    with GlobusComputeExecutor(endpoint_id=compute_endpoint, 
                               user_endpoint_config=POLARIS_CONFIG,
                              ) as remote_exec:
        # Create manager with agents and their assigned executors
        async with await Manager.from_exchange_factory(
            factory=HttpExchangeFactory(),
            executors=remote_exec,
        ) as manager:
            
            # Launch the DrugReasoner and DrugValidator agents on Polaris
            drug_reasoner = await manager.launch(DrugReasoner, args=(api_key,))
            drug_validator = await manager.launch(DrugValidator)

            # 1. Generate Candidates with DrugReasoner
            candidates = await drug_reasoner.generate_candidates(prompt)
            print("Candidates Identified ===================")
            for c in candidates:
                print(f"{c['name']}: {c['smiles']}")
                
            # 2. Validate Candidates with DrugValidator
            print("=========================================")
            print("Validating candidates .... ")                
            validated_candidates = await drug_validator.validate_and_featurize(candidates)
            
            # 3. Estimate Binding Energy with DrugValidator
            print("Estimating binding energies ...")
            simulated_candidates = await drug_validator.estimate_binding_energy(validated_candidates)
            
            # 4. Identify Lead Candidate
            print("Identifying lead candidate ...")
            lead_candidate = await drug_reasoner.select_lead(simulated_candidates)
            
            print(f"Lead candidate: {lead_candidate}")
        remote_exec.shutdown(wait=True)
    return simulated_candidates, lead_candidate


# Launch the campaign
simulated_candidates, lead_candidate = await drug_finder("kinase inhibitor targeting EGFR for NSCLC")

Candidates Identified ===================
Gefitinib analog: COc1ccc2nc(Nc3ccc(F)cc3Cl)nc(N3CCOCC3)c2c1
Erlotinib scaffold: C#Cc1ccc(cc1)NC(=O)C=Cc2nc(Nc3ccc(F)cc3)nc(N)n2
Acrylamide covalent quinazoline: COc1ccc2nc(Nc3ccc(F)cc3)nc(NCC=CC(=O)N)c2c1
Pyrimidine‑type EGFR inhibitor: CN(C)CCOCCc1nc(Nc2ccc(F)cc2)nc2ncccn12
Piperazine quinazoline: Clc1ccc(F)cc1Nc2nc3cc(N4CCNCC4)ccc3n2
Validating candidates .... 
Estimating binding energies ...
Identifying lead candidate ...
Lead candidate: {'lead_name': 'Gefitinib analog', 'reasoning': 'The Gefitinib analog exhibits the lowest strain energy (39.05\u202fkcal/mol), indicating a geometrically relaxed conformation ready for binding. Its physicochemical profile meets Lipinski criteria with a moderate logP (4.0) and low TPSA (59.5\u202fÅ²), supporting good permeability, while retaining a well‑validated quinazoline pharmacophore that balances novelty with proven EGFR activity.'}


## Visualize Results

We use the candidates generated in the above steps, the computed chemical properties and highlight the lead candidate.

In [12]:
from drug_visualize import visualize_results

visualize_results(simulated_candidates, lead_candidate)

ModuleNotFoundError: No module named 'cairosvg'